In [1]:
%pip install torch transformers langchain langchain-community langchain-huggingface \
    sentence-transformers faiss-cpu pypdf accelerate bitsandbytes


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Simple RAG over Computer Science Papers

Runs fully locally on Apple Silicon (MPS) — no cloud APIs, no CUDA.

Stack: `pypdf` (+ `pytesseract` OCR fallback for scanned PDFs) → `langchain` text splitting → `sentence-transformers` embeddings → `faiss` vector store → local Hugging Face model for generation.

In [2]:
import os
from pathlib import Path

import pypdf
import pytesseract
from pdf2image import convert_from_path
from langchain_core.documents import Document

PAPERS_DIR = Path("computer_science_papers")
MIN_CHARS_PER_PAGE = 200  # below this, treat the page as a scan and OCR it

## Hybrid PDF loader

For each page: try `pypdf` text extraction first (fast, high-quality for born-digital PDFs). If a page comes back with almost no text, fall back to rendering that single page to an image and running Tesseract OCR on it. This means clean PDFs never pay the OCR cost, and scanned PDFs (like `Hoare69.pdf`) still produce usable text.

In [3]:
def ocr_page(pdf_path: Path, page_number: int) -> str:
    """Render a single PDF page to an image and OCR it. page_number is 0-indexed."""
    images = convert_from_path(
        str(pdf_path),
        first_page=page_number + 1,
        last_page=page_number + 1,
        dpi=300,
    )
    return pytesseract.image_to_string(images[0])


def load_pdf(pdf_path: Path) -> list[Document]:
    """Load one PDF into a list of per-page LangChain Documents, OCR'ing scanned pages."""
    reader = pypdf.PdfReader(str(pdf_path))
    documents = []

    for page_number, page in enumerate(reader.pages):
        text = (page.extract_text() or "").strip()
        method = "text"

        if len(text) < MIN_CHARS_PER_PAGE:
            ocr_text = ocr_page(pdf_path, page_number).strip()
            if len(ocr_text) > len(text):
                text, method = ocr_text, "ocr"

        if not text:
            continue  # blank page (cover sheets, etc.) — nothing to index

        documents.append(
            Document(
                page_content=text,
                metadata={
                    "source": pdf_path.name,
                    "page": page_number + 1,
                    "extraction_method": method,
                },
            )
        )

    return documents

## Load every paper in the directory

In [4]:
all_documents: list[Document] = []

for pdf_path in sorted(PAPERS_DIR.glob("*.pdf")):
    docs = load_pdf(pdf_path)
    ocr_pages = sum(1 for d in docs if d.metadata["extraction_method"] == "ocr")
    print(f"{pdf_path.name:40s} pages={len(docs):3d}  ocr_pages={ocr_pages}")
    all_documents.extend(docs)

print(f"\nTotal pages loaded: {len(all_documents)}")

APP.pdf                                  pages=  1  ocr_pages=0
Cook1971-retyped.pdf                     pages=  8  ocr_pages=0
Goto-Harmful-Dijkstra.pdf                pages=  3  ocr_pages=0
Hoare69.pdf                              pages=  6  ocr_pages=6
Paper_Codd.pdf                           pages= 11  ocr_pages=0
Rsapaper.pdf                             pages= 15  ocr_pages=0
attention.pdf                            pages= 15  ocr_pages=0
deep-learning-cnn.pdf                    pages=  9  ocr_pages=0
entropy.pdf                              pages= 55  ocr_pages=0
flp.pdf                                  pages=  9  ocr_pages=0
lamport-clocks.pdf                       pages=  8  ocr_pages=0
mapreduce.pdf                            pages= 13  ocr_pages=0
newdirs-crypto-diffie-helman.pdf         pages= 11  ocr_pages=0
p-tr-1971.pdf                            pages= 31  ocr_pages=0
turing36.pdf                             pages= 36  ocr_pages=0

Total pages loaded: 231


## Sanity check

Spot-check that the scanned paper (`Hoare69.pdf`) actually produced usable OCR text, and that a normal paper still used plain extraction.

In [5]:
for doc in all_documents:
    if doc.metadata["source"] == "Hoare69.pdf" and doc.metadata["page"] == 1:
        print(f"--- {doc.metadata} ---")
        print(doc.page_content[:500])
        break

--- {'source': 'Hoare69.pdf', 'page': 1, 'extraction_method': 'ocr'} ---
An Axiomatic Basis for
Computer Programming

C. A. R. Hoare
The Queen’s University of Belfast,* Northern Ireland

In this paper an attempt is made to explore the logical founda-
tions of computer programming by use of techniques which
were first applied in the study of geometry and have later
been extended to other branches of mathematics. This in-
volves the elucidation of sets of axioms and rules of inference
which can be used in proofs of the properties of computer
programs. Examples are give


## Chunk documents

The per-page `Document`s from the loader are too large and too unevenly sized to embed directly — a dense page of a Codd or Shannon paper vs. a sparse title page skews retrieval. Split each page into overlapping chunks with `RecursiveCharacterTextSplitter`, which tries to break on paragraph/sentence boundaries before falling back to raw character splits. The overlap keeps a sentence that straddles a chunk boundary from losing context in either half.

Metadata (`source`, `page`, `extraction_method`) carries over from the parent page automatically, and each chunk gets its own `chunk` index appended.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks: list[Document] = text_splitter.split_documents(all_documents)

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk"] = i

print(f"Pages:  {len(all_documents)}")
print(f"Chunks: {len(chunks)}")
print(f"Avg chunk length: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pages:  231
Chunks: 889
Avg chunk length: 861 chars


## Embeddings

`all-MiniLM-L6-v2` is a small, fast sentence-transformers model — good enough for semantic retrieval over ~15 papers and light enough to embed everything in well under a minute on Apple Silicon. `device="mps"` runs it on the Metal GPU backend instead of CPU.

In [7]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Embedding device: {DEVICE}")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)

Embedding device: mps

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10886.88it/s]


## Build the vector store (FAISS)

Embed every chunk and index it in FAISS — an in-memory, CPU-based similarity index, no server required. We also save it to disk so future notebook runs can load the index instead of re-embedding everything from scratch.

In [8]:
from langchain_community.vectorstores import FAISS

VECTOR_STORE_DIR = "faiss_index"

if Path(VECTOR_STORE_DIR).exists():
    vector_store = FAISS.load_local(
        VECTOR_STORE_DIR, embeddings, allow_dangerous_deserialization=True
    )
    print(f"Loaded existing index from ./{VECTOR_STORE_DIR}")
else:
    vector_store = FAISS.from_documents(chunks, embeddings)
    vector_store.save_local(VECTOR_STORE_DIR)
    print(f"Built and saved new index to ./{VECTOR_STORE_DIR}")

print(f"Vectors in index: {vector_store.index.ntotal}")

Loaded existing index from ./faiss_index
Vectors in index: 889


/var/folders/hy/3v5644z940bgcjdl87wc8k5h0000gn/T/ipykernel_60448/2302048079.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Sanity check — retrieval

Run a test query and confirm the top matches come from the paper we'd expect, with reasonable similarity scores (lower = more similar, since this is L2 distance over normalized embeddings).

In [9]:
test_query = "What is the halting problem and why is it undecidable?"
results = vector_store.similarity_search_with_score(test_query, k=3)

for doc, score in results:
    print(f"[score={score:.4f}] {doc.metadata['source']} (page {doc.metadata['page']})")
    print(doc.page_content[:300].replace("\n", " "))
    print()

[score=1.0606] turing36.pdf (page 17)
the ?n-th figure in a u. Let /? be the sequence with  \—<j> n(n) as its n-th. figure. Since /3 is computable, there exists a number K such that l—cf) ll(n) = <f) K(n) all n.  Putting  n = K, we  have 1  = 2(f> K(K), i.e. 1 is even. This is impossible. The computable sequences are therefore not enume

[score=1.0820] turing36.pdf (page 34)
1936.] ON COMPUTABLE NUMBERS.  263 is interesting to express Un(ii) in a form in which all quantors are at the beginning. Un(At) is, in fact, expressible in the form {u){3x){w){3u1)...{3un)%, (I) where 95 contains no quantors, and n = 6. By unimportant modifications we can obtain a formula, with all

[score=1.0917] turing36.pdf (page 17)
246 A. M.  TURING  [NOV. 12, in«t fl(t(in« 1),tt) «**•  The  next  complete configuration is written down,. a R, E in^t 1(a) carrying out the marked instruc- L) ce 5(o»,.t>, y, x, u, w)  tions - The  letters  u > v> w> x> V are erased.  -^anf. i?) ce 5(o», v, x, u, y, w) \nitx{N) e

## Local generation model

Retrieval is working — now wire up a local model to actually answer questions from the retrieved chunks.

`Qwen2.5-1.5B-Instruct` is a good default here: small enough to load comfortably on any Apple Silicon Mac without quantization, and a capable instruction-follower for a RAG-style "answer using only this context" task. Loaded in `bfloat16` on `mps`. If you have a machine with more memory and want better answer quality, swap `MODEL_NAME` for `Qwen/Qwen2.5-3B-Instruct` or `Qwen/Qwen2.5-7B-Instruct` — same code, just a bigger download and more RAM.

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16).to(DEVICE)

generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    return_full_text=False,
)

llm = HuggingFacePipeline(pipeline=generation_pipeline)
print(f"Loaded {MODEL_NAME} on {DEVICE}")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 8010.77it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Loaded Qwen/Qwen2.5-1.5B-Instruct on mps


## Build the RAG chain

Wire retrieval and generation together with LangChain's expression language (LCEL): retrieve the top chunks for a question, format them into the prompt's `{context}` slot alongside the `{question}`, and pass the whole thing to the local model. The prompt explicitly instructs the model to answer only from the provided context — this is what keeps a small local model from confidently hallucinating instead of saying "not in these papers."

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = PromptTemplate.from_template(
    "You are a helpful assistant answering questions about computer science papers.\n"
    "Use ONLY the following context to answer the question. If the context doesn't "
    "contain the answer, say so instead of guessing.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})


def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(
        f"[{d.metadata['source']} p.{d.metadata['page']}] {d.page_content}" for d in docs
    )


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

## Ask a question

In [12]:
import textwrap

question = "What is the halting problem and why is it undecidable?"

sources = retriever.invoke(question)
print("Retrieved from:")
for doc in sources:
    print(f"  - {doc.metadata['source']} (page {doc.metadata['page']})")

print("\nAnswer:")
print(textwrap.fill(rag_chain.invoke(question).strip(), width=100))

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved from:
  - turing36.pdf (page 17)
  - turing36.pdf (page 34)
  - turing36.pdf (page 17)
  - turing36.pdf (page 2)

Answer:


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The halting problem is a decision problem in computer science where one needs to determine whether a
program will eventually halt or run forever when executed on a given input. It is undecidable
because there is no algorithm that can solve every possible case of the halting problem. The reason
behind this is that the Halting Problem is inherently non-computable due to Gödel's First
Incompleteness Theorem, which states that any consistent formal system capable of expressing basic
arithmetic is incomplete. This means that there are statements within the system that cannot be
proven either true or false, making it impossible to create an algorithm that can always predict the
outcome of running programs on inputs. Therefore, while we can write algorithms to check if certain
specific programs halt on specific inputs, we cannot write an algorithm that works universally for
all programs and inputs. This limitation applies to both Turing Machines and other models of
computation.


## Reusable ask function + simple loop

Wrap retrieval + generation in one helper that also cites which paper(s)/page(s) the answer was drawn from — the citation is computed deterministically from the retrieved chunks' metadata (not asked of the model), so it's always accurate regardless of how well the small model follows a "cite your source" instruction.

In [13]:
test_query = "Why is data getting larger to be called big data?"
results = vector_store.similarity_search_with_score(test_query, k=3)

for doc, score in results:
    print(f"[score={score:.4f}] {doc.metadata['source']} (page {doc.metadata['page']})")
    print(doc.page_content[:300].replace("\n", " "))
    print()

[score=1.2881] mapreduce.pdf (page 10)
Froogle products,  extraction of data used to produce reports of popular queries (e.g. Google Zeitgeist),  extraction of properties of web pages for new exper- iments and products (e.g. extraction of geographi- cal locations from a large corpus of web pages for localized search), and  large-scale

[score=1.3187] mapreduce.pdf (page 12)
of problems are easily expressible as MapReduce com- putations. For example, MapReduce is used for the gen- eration of data for Google's production web search ser- vice, for sorting, for data mining, for machine learning, and many other systems. Third, we have developed an implementation of MapReduc

[score=1.3190] Paper_Codd.pdf (page 1)
Information Retrieval P. BAXENDALE, Editor  A Relational Model of Data for  Large Shared Data Banks  E. F. CODD  IBM Research Laboratory, San Jose, California  Future users of large data banks must be protected from  having to know how the data is organized in the machine (the  

In [14]:
def ask_rag(question: str) -> None:
    retrieved = retriever.invoke(question)
    answer = rag_chain.invoke(question)

    # Cite the paper(s) the answer actually drew on, most-relevant first, de-duplicated by (source, page)
    seen = set()
    citations = []
    for doc in retrieved:
        key = (doc.metadata["source"], doc.metadata["page"])
        if key not in seen:
            seen.add(key)
            citations.append(f"{doc.metadata['source']} (p.{doc.metadata['page']})")

    print(textwrap.fill(f"Q: {question}", width=100))
    print(textwrap.fill(f"A: {answer.strip()}", width=100, subsequent_indent="   "))
    print(textwrap.fill(f"Source: {'; '.join(citations)}", width=100, subsequent_indent="        "))
    print("-" * 100)

## 15-question test set

One simple, direct question per paper in the corpus — an easy way to eyeball whether retrieval consistently pulls from the right source across the whole collection, not just the one or two papers we've been testing against.

In [15]:
test_questions = [
    "What problem does 'A Protocol for Packet Network Intercommunication' aim to solve?",  # APP.pdf
    "What class of problems does Cook prove are NP-complete?",  # Cook1971-retyped.pdf
    "Why does Dijkstra argue the goto statement should be abolished from programming languages?",  # Goto-Harmful-Dijkstra.pdf
    "What is Hoare's approach to proving the correctness of computer programs?",  # Hoare69.pdf
    "What is a relational model of data, according to Codd?",  # Paper_Codd.pdf
    "How does the RSA method produce digital signatures and public-key cryptosystems?",  # Rsapaper.pdf
    "What is the main neural network architecture proposed in 'Attention Is All You Need'?",  # attention.pdf
    "What neural network architecture won the ImageNet classification competition in this paper?",  # deep-learning-cnn.pdf
    "How does Shannon define entropy in his theory of communication?",  # entropy.pdf
    "What do Fischer, Lynch, and Paterson prove about distributed consensus with one faulty process?",  # flp.pdf
    "What is a logical clock used for in Lamport's paper on ordering events?",  # lamport-clocks.pdf
    "What are the two main functions in the MapReduce programming model?",  # mapreduce.pdf
    "What problem in cryptography do Diffie and Hellman address with public-key cryptography?",  # newdirs-crypto-diffie-helman.pdf
    "What criterion does Parnas recommend for decomposing a system into modules?",  # p-tr-1971.pdf
    "What is the Entscheidungsproblem and how does Turing address it?",  # turing36.pdf
]

print(f"{len(test_questions)} test questions")

15 test questions


## Run the batch test

In [16]:
for question in test_questions:
    ask_rag(question)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What problem does 'A Protocol for Packet Network Intercommunication' aim to solve?
A: The paper aims to develop a protocol that supports the sharing of resources across different
   packet switching networks, despite potential differences in implementation details such as
   addressing methods, maximum packet sizes, and time delays. It seeks to facilitate connections
   between existing networks while keeping the interfaces as simple and reliable as possible. The
   goal is to enable conversion between packet switching strategies at the interface rather than at
   the hosts, allowing for efficient data transfer between networks.
Source: APP.pdf (p.1); flp.pdf (p.1)
----------------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What class of problems does Cook prove are NP-complete?
A: According to the provided text, Cook proves that the traveling salesman problem, the
   satisfiability problem for propositional calculus, the knapsack problem, the graph coloring
   problem, and many scheduling and minimization problems are NP-complete. Specifically, the passage
   states:  "Among the problems known to be solvable in NP time, but not known to be solvable in P
   time, are versions of the traveling salesman problem, the satisfiability problem for
   propositional calculus, the knapsack problem, the graph coloring problem, and many scheduling and
   minimization problems [13, pp. 363-4041, [14]."  This indicates that Cook's proof demonstrates
   the NP-completeness of these specific problems.
Source: newdirs-crypto-diffie-helman.pdf (p.10); Cook1971-retyped.pdf (p.3)
----------------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: What are the two main functions in the MapReduce programming model?
A: The two main functions in the MapReduce programming model are the Map function and the Reduce
   function. The Map function takes an input pair and produces a set of intermediate key/value
   pairs, while the Reduce function accepts an intermediate key and a set of values for that key,
   merging these values to form a possibly smaller set of values. Typically, just one output value
   is produced per Reduce invocation. The intermediate values are passed to the user's reduce
   function via an iterator, allowing the programmer to handle large lists of values that cannot fit
   into memory.
Source: mapreduce.pdf (p.1); mapreduce.pdf (p.2)
----------------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What problem in cryptography do Diffie and Hellman address with public-key cryptography?
A: Diffie and Hellman address the problem of securely exchanging cryptographic keys over an insecure
   channel without revealing the actual key to anyone else involved in the communication. They
   propose a method for generating a shared secret key between two parties who wish to communicate
   privately, even when they don't trust each other's intentions or capabilities. Their approach
   involves each party choosing a random number and then computing a product of these numbers along
   with a special "public" number generated by both parties. By exchanging this information
   publicly, the two parties can compute the same shared secret key, allowing them to encrypt and
   decrypt messages safely. This method does not rely on traditional private key encryption methods,
   as it uses a combination of mathematical properties and computational complexity to ensure
   security. Essentially, their

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What criterion does Parnas recommend for decomposing a system into modules?
A: According to D. L. Parnas, two main criteria can be used when decomposing a system into modules:
   1. Make each 'major step' in the processing a module. This means breaking down complex processes
   into smaller, manageable parts based on logical divisions within the overall system.  2. Use
   information hiding as a criterion. Modules do not necessarily need to correspond directly to
   steps in the processing; they can encapsulate data structures or other elements that are relevant
   to specific functionalities without being tied to particular operations.  These recommendations
   suggest that Parnas believes in separating concerns and focusing on the functionality rather than
   just the sequence of tasks, which can lead to better modularity and maintainability of software
   systems. However, he also notes that these methods may become less effective as the complexity of
   the system increases, esp

In [17]:
question = "What is the purpose of AI?"
ask_rag(question)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the purpose of AI?
A: The purpose of AI is to create machines that can perform tasks that would normally require human
   intelligence, such as learning, reasoning, problem-solving, perception, and natural language
   processing. These machines are designed to mimic human cognitive abilities through algorithms and
   computational models. The goal is to develop systems capable of understanding complex
   information, making decisions based on data analysis, and interacting with humans in ways that
   seem intelligent and intuitive. While there have been significant advancements in AI technology
   over the past few decades, challenges remain in areas like ethical considerations, bias, privacy
   concerns, and ensuring safety and reliability in autonomous systems. The field continues to
   evolve rapidly, driven by ongoing research and development efforts aimed at improving AI's
   capabilities across various domains.
Source: Paper_Codd.pdf (p.11); turing36.pdf (p.24); turing